# 03 — Ablation Experiment

**Amaç:** Activation analysis ile seçilen nöronu/feature'ı sıfırlayarak normal ve müdahaleli model davranışını karşılaştırmak.

In [ ]:
import os, sys
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.append(ROOT)
from src.model import build_model
from src.interventions import make_ablation_hook
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(42).to(device)
model.load_state_dict(torch.load('../results/baseline_model.pt', map_location=device))
model.eval()
test = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())
loader = DataLoader(test, batch_size=256, shuffle=False)
selected_neurons = [0, 1, 2]  # Replace with candidates identified in notebook 02
loss_fn = torch.nn.CrossEntropyLoss()
def evaluate_current_model():
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            pred = logits.argmax(1).cpu()
            correct += int((pred == y).sum())
            total += len(y)
    return correct / total
normal_accuracy = evaluate_current_model()
handle = model.net[4].register_forward_hook(make_ablation_hook(selected_neurons))
ablated_accuracy = evaluate_current_model()
handle.remove()
print('selected_neurons:', selected_neurons)
print('normal_accuracy:', normal_accuracy)
print('ablated_accuracy:', ablated_accuracy)
print('accuracy_change:', ablated_accuracy - normal_accuracy)

## Yorum
Normal ve ablated koşul aynı veri üzerinde karşılaştırılır. Etki farklı örneklerde tekrarlandığında causal evidence/support güçlenir; tek koşudan mekanizma kanıtlanmış sayılmaz.